# Task 1: Fine-Tuning Flan-T5-Base for Medical Question Answering

**Student:** Praveen K Gandikota (ID: 16829772)

**Module:** 7043SCN - Generative AI and Reinforcement Learning

**Model:** google/flan-t5-base (248M params, encoder-decoder)

**Dataset:** AnonymousSub/MedQuAD_47441_Question_Answer_Pairs

**Technique:** LoRA (Low-Rank Adaptation)

In [ ]:
import subprocess, sys
for pkg in ["torch", "transformers>=4.36.0", "datasets>=2.16.0", "peft>=0.7.0",
            "accelerate>=0.25.0", "evaluate>=0.4.0", "rouge-score", "nltk",
            "sentencepiece", "protobuf", "scikit-learn", "matplotlib", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

In [ ]:
import os, time, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import torch
import nltk
from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CONFIG = {
    "model_name": "google/flan-t5-base",
    "dataset_name": "AnonymousSub/MedQuAD_47441_Question_Answer_Pairs",
    "max_input_length": 512, "max_target_length": 256,
    "batch_size": 8, "learning_rate": 3e-4, "num_epochs": 3,
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.1,
    "output_dir": "./results", "seed": SEED,
    "num_samples": 5000,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
dataset = load_dataset(CONFIG["dataset_name"])
full = dataset["train"].filter(lambda x: x["Answers"] is not None and len(x["Answers"].strip()) > 0)
print(f"Dataset: {len(dataset['train'])} total, {len(full)} with answers")
print(f"Sample Q: {full[0]['Questions'][:120]}")
print(f"Sample A: {full[0]['Answers'][:120]}")

full = full.shuffle(seed=SEED).select(range(min(CONFIG["num_samples"], len(full))))
n = len(full)
split_dataset = DatasetDict({
    "train": full.select(range(int(n*0.8))),
    "validation": full.select(range(int(n*0.8), int(n*0.9))),
    "test": full.select(range(int(n*0.9), n)),
})
print(f"Train: {len(split_dataset['train'])}, Val: {len(split_dataset['validation'])}, Test: {len(split_dataset['test'])}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def preprocess(examples):
    inputs = [f"Answer the medical question: {q}" for q in examples["Questions"]]
    model_inputs = tokenizer(inputs, max_length=CONFIG["max_input_length"], truncation=True, padding="max_length")
    labels = tokenizer(examples["Answers"], max_length=CONFIG["max_target_length"], truncation=True, padding="max_length")
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = split_dataset.map(preprocess, batched=True, remove_columns=split_dataset["train"].column_names)
print("Tokenization complete.")

In [ ]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    bleu_scores = []
    for p, r in zip(decoded_preds, decoded_labels):
        try:
            pt, rt = p.split(), r.split()
            bleu_scores.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
        except Exception:
            bleu_scores.append(0.0)
    return {k: round(v, 4) for k, v in {"rouge1": rouge["rouge1"], "rouge2": rouge["rouge2"], "rougeL": rouge["rougeL"], "bleu": np.mean(bleu_scores)}.items()}

## Baseline Evaluation (Zero-Shot)

In [ ]:
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"]).to(device)
model_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Model parameters: {model_params:,} ({model_params/1e6:.1f}M)")

baseline_preds, baseline_refs, test_questions = [], [], []
baseline_model.eval()
t0 = time.time()
with torch.no_grad():
    for i, sample in enumerate(split_dataset["test"]):
        inp = tokenizer(f"Answer the medical question: {sample['Questions']}", return_tensors="pt",
                        max_length=CONFIG["max_input_length"], truncation=True).to(device)
        out = baseline_model.generate(**inp, max_new_tokens=CONFIG["max_target_length"], num_beams=4, early_stopping=True)
        baseline_preds.append(tokenizer.decode(out[0], skip_special_tokens=True).strip())
        baseline_refs.append(sample["Answers"].strip())
        test_questions.append(sample["Questions"])
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(split_dataset['test'])}")
baseline_eval_time = time.time() - t0

bl_rouge = rouge_metric.compute(predictions=baseline_preds, references=baseline_refs, use_stemmer=True)
bl_bleu = []
for p, r in zip(baseline_preds, baseline_refs):
    try:
        pt, rt = p.split(), r.split()
        bl_bleu.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
    except: bl_bleu.append(0.0)

baseline_metrics = {"rouge1": round(bl_rouge["rouge1"],4), "rouge2": round(bl_rouge["rouge2"],4),
                    "rougeL": round(bl_rouge["rougeL"],4), "bleu": round(np.mean(bl_bleu),4)}
print(f"Baseline: R1={baseline_metrics['rouge1']}, R2={baseline_metrics['rouge2']}, RL={baseline_metrics['rougeL']}, BLEU={baseline_metrics['bleu']}")

for i in range(min(3, len(test_questions))):
    print(f"\nQ: {test_questions[i][:150]}")
    print(f"Pred: {baseline_preds[i][:150]}")
    print(f"Ref: {baseline_refs[i][:150]}")

del baseline_model
if torch.cuda.is_available(): torch.cuda.empty_cache()

## Hardware Assessment

Full fine-tuning of Flan-T5-Base (248M parameters) requires updating all parameters simultaneously, demanding approximately 8-12 GB of GPU memory for batch processing with gradients. Our available hardware (CPU-only, no dedicated GPU) makes full fine-tuning **infeasible** due to excessive training time (estimated 20+ hours for 3 epochs). Therefore, we use **LoRA (Low-Rank Adaptation)** which reduces trainable parameters by ~99%, making training feasible on CPU within 1-2 hours.

## LoRA Fine-Tuning

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])
lora_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=CONFIG["lora_r"],
                         lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
                         target_modules=["q", "v"], bias="none")
model = get_peft_model(model, lora_config)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
reduction = (1 - trainable/total_params) * 100
print(f"LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, targets=[q, v]")
print(f"Total: {total_params:,}, Trainable: {trainable:,} ({trainable/total_params*100:.4f}%), Reduction: {reduction:.2f}%")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"], num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"], per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"], weight_decay=0.01, warmup_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="rougeL", greater_is_better=True,
    predict_with_generate=True, generation_max_length=CONFIG["max_target_length"],
    logging_steps=50, report_to="none", seed=SEED, fp16=torch.cuda.is_available(), dataloader_num_workers=0,
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    processing_class=tokenizer, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training: {CONFIG['num_epochs']} epochs, batch={CONFIG['batch_size']}, lr={CONFIG['learning_rate']}")
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
print(f"Done in {train_time:.1f}s ({train_time/60:.1f} min), loss={train_result.training_loss:.4f}")

epoch_metrics = [e for e in trainer.state.log_history if "eval_rouge1" in e]
for em in epoch_metrics:
    print(f"  Epoch {em['epoch']:.0f}: Loss={em['eval_loss']:.4f}, R1={em['eval_rouge1']:.4f}, RL={em['eval_rougeL']:.4f}, BLEU={em['eval_bleu']:.4f}")

## LoRA Evaluation on Test Set

In [ ]:
lora_preds = []
model.eval()
t0 = time.time()
with torch.no_grad():
    for i, sample in enumerate(split_dataset["test"]):
        inp = tokenizer(f"Answer the medical question: {sample['Questions']}", return_tensors="pt",
                        max_length=CONFIG["max_input_length"], truncation=True).to(device)
        out = model.generate(**inp, max_new_tokens=CONFIG["max_target_length"], num_beams=4, early_stopping=True)
        lora_preds.append(tokenizer.decode(out[0], skip_special_tokens=True).strip())
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(split_dataset['test'])}")
lora_eval_time = time.time() - t0

lora_rouge = rouge_metric.compute(predictions=lora_preds, references=baseline_refs, use_stemmer=True)
lora_bleu = []
for p, r in zip(lora_preds, baseline_refs):
    try:
        pt, rt = p.split(), r.split()
        lora_bleu.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
    except: lora_bleu.append(0.0)

lora_metrics = {"rouge1": round(lora_rouge["rouge1"],4), "rouge2": round(lora_rouge["rouge2"],4),
                "rougeL": round(lora_rouge["rougeL"],4), "bleu": round(np.mean(lora_bleu),4)}
print(f"LoRA: R1={lora_metrics['rouge1']}, R2={lora_metrics['rouge2']}, RL={lora_metrics['rougeL']}, BLEU={lora_metrics['bleu']}")

for i in range(min(3, len(test_questions))):
    print(f"\nQ: {test_questions[i][:150]}")
    print(f"LoRA: {lora_preds[i][:150]}")
    print(f"Ref: {baseline_refs[i][:150]}")

## Comparative Analysis and Visualization

In [ ]:
import pandas as pd
comparison = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"],
    "Baseline": [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"]],
    "LoRA": [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"]],
})
comparison["Improvement"] = comparison["LoRA"] - comparison["Baseline"]
comparison["Improvement_%"] = ((comparison["Improvement"] / comparison["Baseline"].replace(0,1)) * 100).round(2)
print(comparison.to_string(index=False))
print(f"\nTotal params: {total_params:,}, Trainable: {trainable:,}, Reduction: {reduction:.2f}%")
print(f"Training: {train_time:.1f}s, Baseline eval: {baseline_eval_time:.1f}s, LoRA eval: {lora_eval_time:.1f}s")

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
metrics_names = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
bl_vals = [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"]]
lo_vals = [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"]]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

x = np.arange(len(metrics_names)); w = 0.35
axes[0,0].bar(x-w/2, bl_vals, w, label="Baseline", color="#3498db", edgecolor="black", linewidth=0.5)
axes[0,0].bar(x+w/2, lo_vals, w, label="LoRA", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(metrics_names)
axes[0,0].set_title("Baseline vs LoRA"); axes[0,0].legend(); axes[0,0].grid(axis="y", alpha=0.3)

if epoch_metrics:
    eps = [e["epoch"] for e in epoch_metrics]
    axes[0,1].plot(eps, [e["eval_loss"] for e in epoch_metrics], "b-o", lw=2)
    axes[0,1].set_title("Validation Loss"); axes[0,1].set_xlabel("Epoch"); axes[0,1].grid(alpha=0.3)
    axes[1,0].plot(eps, [e["eval_rouge1"] for e in epoch_metrics], "r-o", lw=2, label="R1")
    axes[1,0].plot(eps, [e["eval_rouge2"] for e in epoch_metrics], "g-s", lw=2, label="R2")
    axes[1,0].plot(eps, [e["eval_rougeL"] for e in epoch_metrics], "b-^", lw=2, label="RL")
    axes[1,0].set_title("ROUGE Scores"); axes[1,0].set_xlabel("Epoch"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].bar(["Total", "Trainable (LoRA)"], [total_params, trainable], color=["#3498db", "#e74c3c"], edgecolor="black")
axes[1,1].set_title("Parameter Efficiency"); axes[1,1].set_yscale("log"); axes[1,1].grid(axis="y", alpha=0.3)

plt.suptitle("Flan-T5-Base Medical Q&A: LoRA Fine-Tuning Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/results.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
results = {
    "student": "Praveen K Gandikota (16829772)", "config": CONFIG,
    "baseline_metrics": baseline_metrics, "lora_metrics": lora_metrics,
    "training_time_seconds": train_time, "total_parameters": total_params,
    "trainable_parameters": trainable, "parameter_reduction_percent": round(reduction, 2),
    "epoch_metrics": epoch_metrics,
}
with open(f"{CONFIG['output_dir']}/results.json", "w") as f:
    json.dump(results, f, indent=2)
comparison.to_csv(f"{CONFIG['output_dir']}/comparison.csv", index=False)
model.save_pretrained(f"{CONFIG['output_dir']}/lora_model")
tokenizer.save_pretrained(f"{CONFIG['output_dir']}/lora_model")
print(f"All results saved to {CONFIG['output_dir']}/")